# 13 Ensemble check on the saved scores

Does combining the six models beat the best single model? This notebook only reads the scores saved by notebooks
04 to 09 (`results/preds/`); it trains nothing and writes nothing to `results/log.csv`. It is an exploratory check
for the group's decision, not part of the 6 × 4 grid in `Context.md`.

**Rules that keep the check honest**

- Every choice (which blends, and their weights) is made on the **time-CV fold scores** (days 31 to 141, saved with
  `SAVE_OOF`). The validation block is used once, at the end, to check the blends already fixed.
- Scores are turned into ranks before blending (within each fold, or within the validation block), so models whose
  probabilities sit on different scales combine fairly.
- A **segment blend** uses one set of weights for **known** customers (`uid` seen in the rows the model was trained
  on) and another for **unknown** ones. Whether a customer is known needs no labels, so it is available when a
  transaction arrives (`src/segments.py`).
- The search covers the four strongest models; weights move in steps of `STEP` and sum to 1 (`src/ensemble.py`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score

from src import ensemble, experiment, feature_sets as fs, segments
from src.split import holdout_indices

MODELS = ["logreg", "rf", "lgbm", "xgboost", "catboost", "mlp"]
LABELS = {"logreg": "Logistic regression", "rf": "Random forest", "lgbm": "LightGBM",
          "xgboost": "XGBoost", "catboost": "CatBoost", "mlp": "MLP"}
CANDIDATES = ["xgboost", "lgbm", "catboost", "rf"]   # the four strongest models on the fold scores (section 3)
FEATURE_SET = "G3"         # one set for every model, so the blend combines models, not feature sets
STEP = 0.1                 # weight step; 0.1 gives 286 weightings of four models
SEEDS = [0, 1, 2]
TOP_N = 10                 # rows shown from each weight search
N_BOOT = 200               # bootstrap resamples for the validation intervals
PREDS_DIR = experiment.PREDS_DIR
pd.set_option("display.width", 160)

## 1. Data

In [2]:
spec = fs.load_spec()
ids = pd.read_parquet(ROOT / spec["parquet"],
                      columns=["TransactionID", "TransactionDT", "TransactionDay", "uid", "isFraud"])
train_idx, val_idx, _ = holdout_indices(ids)
ids = ids.iloc[:val_idx[-1] + 1]                       # the test block is not used

oof = ensemble.oof_table(PREDS_DIR, MODELS, FEATURE_SET, ids, train_idx)
known = oof["known"].to_numpy()
print(f"fold scores: {len(oof):,} rows in {oof['fold'].nunique()} folds, {int(oof['isFraud'].sum()):,} fraud; "
      f"known customers {known.mean():.1%} of rows, {int(oof.loc[known, 'isFraud'].sum()):,} fraud")

fold scores: 338,093 rows in 4 folds, 13,198 fraud; known customers 40.0% of rows, 2,989 fraud


## 2. How much the models agree

Spearman correlation of the fold scores (ranked within fold). Two models that rank rows almost identically gain
little from being averaged; a blend helps most when a strong model is paired with a different one.

In [3]:
corr = oof[MODELS].corr(method="spearman").rename(index=LABELS, columns=LABELS)
display(corr.round(3))

,Logistic regression,Random forest,LightGBM,XGBoost,CatBoost,MLP
Logistic regression,1.000,0.578,0.608,0.580,0.777,0.740
Random forest,0.578,1.000,0.707,0.635,0.698,0.507
LightGBM,0.608,0.707,1.000,0.897,0.738,0.610
XGBoost,0.580,0.635,0.897,1.000,0.688,0.612
CatBoost,0.777,0.698,0.738,0.688,1.000,0.675
MLP,0.740,0.507,0.610,0.612,0.675,1.000


## 3. Single models on the fold scores

Mean PR-AUC over the four folds, on all rows and on each segment (known means the `uid` appears in that fold's
training months).

In [4]:
single = pd.DataFrame({
    "all rows": {m: ensemble.fold_pr_auc(oof["isFraud"], oof[m], oof["fold"]) for m in MODELS},
    "known": {m: ensemble.fold_pr_auc(oof.loc[known, "isFraud"], oof.loc[known, m], oof.loc[known, "fold"]) for m in MODELS},
    "unknown": {m: ensemble.fold_pr_auc(oof.loc[~known, "isFraud"], oof.loc[~known, m], oof.loc[~known, "fold"]) for m in MODELS},
}).sort_values("all rows", ascending=False)
display(single.rename(index=LABELS).round(4))

,all rows,known,unknown
XGBoost,0.6377,0.7835,0.5983
LightGBM,0.6343,0.7186,0.6163
CatBoost,0.5912,0.4995,0.6223
Random forest,0.5847,0.6293,0.5850
MLP,0.4828,0.3572,0.5207
Logistic regression,0.4110,0.2380,0.4650


## 4. One set of weights for every row

Every weighting of the candidates, scored by mean fold PR-AUC; the best `TOP_N` are shown. The best fold score is
optimistic (it is the best of many tries); section 6 gives the honest number.

In [5]:
glob = ensemble.search(oof, CANDIDATES, step=STEP)
w_global = ensemble.best_weights(glob, CANDIDATES)
print(f"{len(glob)} weightings tried; best {w_global} -> fold PR-AUC {glob['oof_pr_auc'].iloc[0]:.4f}")
display(glob.head(TOP_N).round(4))

286 weightings tried; best {'xgboost': 0.5, 'lgbm': 0.4, 'catboost': 0.0, 'rf': 0.1} -> fold PR-AUC 0.6430


,xgboost,lgbm,catboost,rf,oof_pr_auc
0,0.5,0.4,0.0,0.1,0.6430
1,0.6,0.3,0.0,0.1,0.6428
2,0.4,0.5,0.0,0.1,0.6428
3,0.7,0.2,0.0,0.1,0.6424
4,0.3,0.6,0.0,0.1,0.6424
5,0.4,0.4,0.0,0.2,0.6423
6,0.5,0.3,0.0,0.2,0.6423
7,0.6,0.2,0.0,0.2,0.6419
8,0.3,0.5,0.0,0.2,0.6419
9,0.8,0.1,0.0,0.1,0.6417


## 5. Separate weights for known and unknown customers

The same search, run once on known rows and once on unknown rows. If the top weightings differ little in score but
agree on which model carries most weight, the result does not hinge on the exact weights.

In [6]:
tab_known = ensemble.search(oof, CANDIDATES, step=STEP, mask=known)
tab_unknown = ensemble.search(oof, CANDIDATES, step=STEP, mask=~known)
w_known = ensemble.best_weights(tab_known, CANDIDATES)
w_unknown = ensemble.best_weights(tab_unknown, CANDIDATES)

for name, tab, mask in (("known", tab_known, known), ("unknown", tab_unknown, ~known)):
    alone = ensemble.fold_pr_auc(oof.loc[mask, "isFraud"], oof.loc[mask, "xgboost"], oof.loc[mask, "fold"])
    print(f"{name}: best {ensemble.best_weights(tab, CANDIDATES)} -> fold PR-AUC {tab['oof_pr_auc'].iloc[0]:.4f} "
          f"(XGBoost alone {alone:.4f}); top {TOP_N} span {tab['oof_pr_auc'].iloc[TOP_N - 1]:.4f} to "
          f"{tab['oof_pr_auc'].iloc[0]:.4f}")
    display(tab.head(TOP_N).round(4))

seg_oof = ensemble.fold_pr_auc(oof["isFraud"], ensemble.segment_blend(oof, known, w_known, w_unknown), oof["fold"])
print(f"segment blend on all fold rows: {seg_oof:.4f}")

known: best {'xgboost': 1.0, 'lgbm': 0.0, 'catboost': 0.0, 'rf': 0.0} -> fold PR-AUC 0.7835 (XGBoost alone 0.7835); top 10 span 0.7614 to 0.7835


,xgboost,lgbm,catboost,rf,oof_pr_auc
0,1.0,0.0,0.0,0.0,0.7835
1,0.9,0.1,0.0,0.0,0.7795
2,0.9,0.0,0.0,0.1,0.7755
3,0.8,0.2,0.0,0.0,0.7750
4,0.8,0.1,0.0,0.1,0.7713
5,0.7,0.3,0.0,0.0,0.7700
6,0.7,0.2,0.0,0.1,0.7666
7,0.8,0.0,0.0,0.2,0.7656
8,0.6,0.4,0.0,0.0,0.7647
9,0.7,0.1,0.0,0.2,0.7614


unknown: best {'xgboost': 0.1, 'lgbm': 0.2, 'catboost': 0.6, 'rf': 0.1} -> fold PR-AUC 0.6353 (XGBoost alone 0.5983); top 10 span 0.6344 to 0.6353


,xgboost,lgbm,catboost,rf,oof_pr_auc
0,0.1,0.2,0.6,0.1,0.6353
1,0.1,0.3,0.5,0.1,0.6353
2,0.0,0.3,0.6,0.1,0.6350
3,0.0,0.4,0.5,0.1,0.6350
4,0.1,0.2,0.5,0.2,0.6348
5,0.0,0.3,0.5,0.2,0.6346
6,0.1,0.1,0.7,0.1,0.6346
7,0.1,0.3,0.4,0.2,0.6345
8,0.1,0.4,0.4,0.1,0.6345
9,0.2,0.2,0.5,0.1,0.6344


segment blend on all fold rows: 0.6694


### 5.1 Fold by fold

The fold scores cover four separate periods (days 31–60, 61–90, 91–120, 121–141). A blend whose advantage is real
should beat XGBoost alone in most of them, not just on the average.

- **Same weights, per fold:** PR-AUC in each fold with the weights from sections 4 and 5. Those weights were chosen
  on the average over these same folds, so this shows consistency, not independent evidence.
- **Weights chosen without the fold (leave one fold out):** for each fold, the segment weights are searched on the
  other three folds only and then scored on the held-out fold. This is closer to independent evidence: the scored
  period played no part in choosing the weights.

In [7]:
FOLDS = sorted(oof["fold"].unique())

def per_fold(score, mask=None):
    m = np.ones(len(oof), bool) if mask is None else mask
    return [average_precision_score(oof.loc[m & (oof["fold"] == k).to_numpy(), "isFraud"],
                                    np.asarray(score)[m & (oof["fold"] == k).to_numpy()]) for k in FOLDS]

xgb = oof["xgboost"].to_numpy()
seg = ensemble.segment_blend(oof, known, w_known, w_unknown)
same = pd.DataFrame({
    "XGBoost alone": per_fold(xgb),
    "Best single weights (s4)": per_fold(ensemble.blend(oof, w_global)),
    "Segment blend (s5)": per_fold(seg),
    "XGBoost alone, unknown": per_fold(xgb, ~known),
    "Segment blend, unknown": per_fold(seg, ~known),
}, index=pd.Index(FOLDS, name="fold"))
same["segment − XGBoost"] = same["Segment blend (s5)"] - same["XGBoost alone"]
print(f"Same weights: the segment blend beats XGBoost alone in {(same['segment − XGBoost'] > 0).sum()} of "
      f"{len(FOLDS)} folds")
display(same.round(4))

loo = []
for k in FOLDS:
    rest = (oof["fold"] != k).to_numpy()
    wk = ensemble.best_weights(ensemble.search(oof, CANDIDATES, step=STEP, mask=rest & known), CANDIDATES)
    wu = ensemble.best_weights(ensemble.search(oof, CANDIDATES, step=STEP, mask=rest & ~known), CANDIDATES)
    held = (oof["fold"] == k).to_numpy()
    s = ensemble.segment_blend(oof[held], known[held], wk, wu)
    y = oof.loc[held, "isFraud"]
    loo.append({"fold": k, "known weights": wk, "unknown weights": wu,
                "XGBoost alone": average_precision_score(y, xgb[held]),
                "segment blend (weights from other folds)": average_precision_score(y, s)})
loo = pd.DataFrame(loo).set_index("fold")
loo["gain"] = loo["segment blend (weights from other folds)"] - loo["XGBoost alone"]
print(f"Leave one fold out: the segment blend beats XGBoost alone in {(loo['gain'] > 0).sum()} of {len(FOLDS)} "
      f"folds; mean gain {loo['gain'].mean():.4f}")
with pd.option_context("display.max_colwidth", 80):
    display(loo.round(4))

Same weights: the segment blend beats XGBoost alone in 4 of 4 folds


,XGBoost alone,Best single weights (s4),Segment blend (s5),"XGBoost alone, unknown","Segment blend, unknown",segment − XGBoost
fold,,,,,,
1,0.5985,0.6018,0.6319,0.5600,0.5991,0.0334
2,0.6573,0.6601,0.6833,0.6272,0.6557,0.0260
3,0.5853,0.6062,0.6325,0.5630,0.6151,0.0473
4,0.7099,0.7038,0.7300,0.6431,0.6713,0.0201


Leave one fold out: the segment blend beats XGBoost alone in 4 of 4 folds; mean gain 0.0314


,known weights,unknown weights,XGBoost alone,segment blend (weights from other folds),gain
fold,,,,,
1,"{'xgboost': 1.0, 'lgbm': 0.0, 'catboost': 0.0, 'rf': 0.0}","{'xgboost': 0.1, 'lgbm': 0.3, 'catboost': 0.5, 'rf': 0.1}",0.5985,0.6307,0.0323
2,"{'xgboost': 1.0, 'lgbm': 0.0, 'catboost': 0.0, 'rf': 0.0}","{'xgboost': 0.1, 'lgbm': 0.2, 'catboost': 0.6, 'rf': 0.1}",0.6573,0.6833,0.0260
3,"{'xgboost': 1.0, 'lgbm': 0.0, 'catboost': 0.0, 'rf': 0.0}","{'xgboost': 0.1, 'lgbm': 0.2, 'catboost': 0.6, 'rf': 0.1}",0.5853,0.6325,0.0473
4,"{'xgboost': 1.0, 'lgbm': 0.0, 'catboost': 0.0, 'rf': 0.0}","{'xgboost': 0.1, 'lgbm': 0.2, 'catboost': 0.6, 'rf': 0.1}",0.7099,0.7300,0.0201


## 6. Check on the validation block (once)

The blends below were fixed in sections 4 and 5 before this cell reads any validation score. For each seed, every
model's seed-s validation scores are ranked and blended; the table gives the mean ± sample sd over seeds 0, 1, 2,
and PR-AUC by segment with a 95% bootstrap interval over rows (`N_BOOT` resamples; the interval is for the mean
over seeds). XGBoost on G1, the best single cell in `10_model_comparison`, is included for reference.

In [8]:
BLENDS = {
    "XGBoost alone (G3)": ("global", {"xgboost": 1.0}),
    "XGBoost + LightGBM, equal": ("global", {"xgboost": 0.5, "lgbm": 0.5}),
    "Four models, equal": ("global", {m: 1 / len(CANDIDATES) for m in CANDIDATES}),
    "Best single weights (section 4)": ("global", w_global),
    "Segment blend (section 5)": ("segment", (w_known, w_unknown)),
}

def val_scores(kind, w, seed):
    if kind == "single_G1":
        t = ensemble.val_table(PREDS_DIR, ["xgboost"], "G1", ids, train_idx, seed)
        return t, t["xgboost"].to_numpy()
    t = ensemble.val_table(PREDS_DIR, CANDIDATES, FEATURE_SET, ids, train_idx, seed)
    s = ensemble.segment_blend(t, t["known"], *w) if kind == "segment" else ensemble.blend(t, w)
    return t, s

rows, seg_rows = [], []
for name, (kind, w) in {**BLENDS, "XGBoost alone (G1, best single cell)": ("single_G1", None)}.items():
    runs = [val_scores(kind, w, s) for s in SEEDS]
    t0 = runs[0][0]
    for t, _ in runs[1:]:
        if not t["TransactionID"].equals(t0["TransactionID"]):
            raise ValueError(f"{name}: seeds cover different rows")
    y, k, scores = t0["isFraud"].to_numpy(), t0["known"].to_numpy(), [s for _, s in runs]
    vals = [average_precision_score(y, s) for s in scores]
    rows.append({"blend": name, "val PR-AUC": f"{np.mean(vals):.4f} ± {np.std(vals, ddof=1):.4f}", "mean": np.mean(vals)})
    seg = segments.segment_pr_auc(y, scores, k, n_boot=N_BOOT)
    seg_rows.append(seg.assign(blend=name))

val = pd.DataFrame(rows).set_index("blend")
seg = pd.concat(seg_rows)
seg["cell"] = seg.apply(lambda r: f"{r.pr_auc:.4f} [{r.low:.3f}, {r.high:.3f}]", axis=1)
val = val.join(seg.pivot(index="blend", columns="segment", values="cell"))
print(f"validation: {len(y):,} rows, {int(y.sum()):,} fraud; known {k.mean():.1%} of rows")
display(val.drop(columns="mean"))

validation: 59,054 rows, 1,851 fraud; known 48.6% of rows


,val PR-AUC,known,unknown
blend,,,
XGBoost alone (G3),0.6434 ± 0.0032,"0.8414 [0.811, 0.869]","0.5369 [0.515, 0.568]"
"XGBoost + LightGBM, equal",0.6390 ± 0.0041,"0.8236 [0.792, 0.852]","0.5485 [0.526, 0.581]"
"Four models, equal",0.6049 ± 0.0013,"0.7358 [0.700, 0.769]","0.5588 [0.534, 0.589]"
Best single weights (section 4),0.6409 ± 0.0035,"0.8234 [0.792, 0.853]","0.5535 [0.530, 0.585]"
Segment blend (section 5),0.6551 ± 0.0009,"0.8414 [0.811, 0.869]","0.5554 [0.530, 0.585]"
"XGBoost alone (G1, best single cell)",0.6495 ± 0.0025,"0.8427 [0.811, 0.871]","0.5427 [0.519, 0.575]"


## 7. Findings

- **Agreement (section 2).** XGBoost and LightGBM rank rows almost the same way (Spearman 0.897), so averaging the
  two adds little. CatBoost and random forest differ more from XGBoost (0.688 and 0.635) but are weaker on the fold
  scores (0.5912 and 0.5847, against 0.6377 for XGBoost).
- **One set of weights (sections 4 and 6).** The best weighting on the fold scores, XGBoost 0.5, LightGBM 0.4,
  random forest 0.1, beats XGBoost alone there (0.6430 vs 0.6377) but not on validation (0.6409 vs 0.6434). An equal
  XGBoost + LightGBM blend scores 0.6390 and an equal four-model blend 0.6049. Blending lifts unknown customers
  slightly (up to 0.5588 against 0.5369) but costs more on known ones (0.7358 to 0.8236 against 0.8414), so a single
  set of weights does not help.
- **Segment weights (section 5).** The models are strong on different customers: on the fold scores XGBoost is best
  on known customers (0.7835) and CatBoost on unknown ones (0.6223). The search gives known customers XGBoost alone,
  and unknown customers CatBoost 0.6, LightGBM 0.2, XGBoost 0.1, random forest 0.1 (fold PR-AUC 0.6353, against
  0.5983 for XGBoost alone). The top of the unknown search is flat: the ten best weightings span 0.6344 to 0.6353 and
  all give CatBoost 0.4 to 0.7, so the result rests on CatBoost carrying most weight for unknown customers, not on
  the exact weights. The known search falls away from XGBoost alone (0.7835 to 0.7614 over the top ten).
- **Fold by fold (section 5.1).** The segment blend beats XGBoost alone in all four folds with the same weights
  (by 0.0201 to 0.0473), and also when each fold's weights are chosen on the other three folds only (4 of 4, mean
  gain 0.0314). The leave-one-fold-out searches pick the same weights every time (known: XGBoost alone; unknown:
  CatBoost 0.5 or 0.6, LightGBM 0.2 or 0.3, XGBoost 0.1, random forest 0.1), so the advantage is consistent across
  periods and does not depend on which period the weights were chosen on. In every fold the gain is on unknown
  customers (for example 0.5630 → 0.6151 in fold 3).
- **Validation check (section 6).** The segment blend scores 0.6551 ± 0.0009, above XGBoost on G3 (0.6434) and the
  best single cell, XGBoost on G1 (0.6495), by 0.0056. Its known-customer score equals XGBoost's (0.8414, the same
  model); all of the gain is on unknown customers (0.5554 [0.530, 0.585] against 0.5427 [0.519, 0.575] for XGBoost on
  G1). The segment blend was also the top option on the fold scores (0.6694), so the validation block confirmed a
  choice made without it. The validation gain is smaller than on the folds, and its intervals overlap, so the size of
  the gain on new data is uncertain; its direction has held in every period checked so far.
- **For the group's decision.** A segment blend is cheap to add, because Step 19 scores every model on the test
  block anyway; it would need its weights fixed as above, a threshold chosen on validation, and its own rows in the
  log, before the experiment freeze. Without it, these results can be reported as an exploratory finding: the
  models' strengths split by customer type, with CatBoost better on new customers and XGBoost on returning ones.